In [1]:
from pathlib import Path

import gc
import json
import math
import os
import random
import re
import time
import warnings

import nibabel as nib
import numpy as np
import pandas as pd

from scipy.ndimage import (
    binary_closing,
    binary_dilation,
    binary_erosion,
    binary_opening,
    generate_binary_structure,
    shift as ndi_shift,
)
from scipy.stats import pearsonr, spearmanr

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


In [2]:
PROJECT_ROOT = Path.cwd()

DATA_DIR = (
    PROJECT_ROOT
    / "Data"
    / "ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
)

DATA_SPLIT_JSON = (
    PROJECT_ROOT
    / "data_split.json"
)

EVALUATION_ROOT = (
    PROJECT_ROOT
    / "evaluation_200"
)

EVALUATION_SUBJECTS_JSON = (
    EVALUATION_ROOT
    / "conditions"
    / "evaluation_subjects_200.json"
)

CONDITION_MASK_DIR = (
    EVALUATION_ROOT
    / "conditions"
    / "masks"
)

MODEL_DIRS = {
    "ddpm_v5":
        EVALUATION_ROOT
        / "ddpm_v5",

    "conditional_ddpm_v3":
        EVALUATION_ROOT
        / "conditional_ddpm_v3",

    "conditional_ldm_v4":
        EVALUATION_ROOT
        / "conditional_ldm_v4",
}

MODEL_PREFIXES = {
    "ddpm_v5":
        "ddpm_v5",

    "conditional_ddpm_v3":
        "conditional_ddpm_v3",

    "conditional_ldm_v4":
        "conditional_ldm_v4",
}

MODEL_DISPLAY_NAMES = {
    "ddpm_v5":
        "DDPM V5",

    "conditional_ddpm_v3":
        "Conditional DDPM V3",

    "conditional_ldm_v4":
        "Conditional LDM V4",
}


# ============================================================
# Independent downstream replicate
# ============================================================

RUN_LABEL = "v3"

SPLIT_SEED = 2026
TRAINING_SEED = 2028
PERTURBATION_SEED = 2026
BOOTSTRAP_SEED = 2026


# The first completed run remains untouched in:
#   downstream_results/
#   downstream_checkpoints/
#   downstream_cache/
#
# This replicate writes only to downstream_v3/.

LEGACY_DOWNSTREAM_ROOT = (
    PROJECT_ROOT
    / "downstream_results"
)

SHARED_CACHE_ROOT = (
    PROJECT_ROOT
    / "downstream_cache"
)

RUN_ROOT = (
    PROJECT_ROOT
    / "downstream_v3"
)

DOWNSTREAM_ROOT = (
    RUN_ROOT
    / "results"
)

CACHE_ROOT = (
    RUN_ROOT
    / "cache"
)

CHECKPOINT_ROOT = (
    RUN_ROOT
    / "checkpoints"
)


# These preprocessed image caches do not depend on training seed.
# They are read from the completed first run.
REAL_LOWRES_DIR = (
    SHARED_CACHE_ROOT
    / "real_lowres"
)

SYNTHETIC_LOWRES_ROOT = (
    SHARED_CACHE_ROOT
    / "synthetic_lowres"
)


# These outputs depend on the new evaluator and are isolated.
QUALITY_BASE_PRED_DIR = (
    CACHE_ROOT
    / "quality_base_predictions"
)

SYNTHETIC_PRED_LOWRES_ROOT = (
    CACHE_ROOT
    / "synthetic_predictions_lowres"
)

PREDICTED_MASK_ROOT = (
    DOWNSTREAM_ROOT
    / "predicted_whole_tumour_masks"
)


for directory in [
    RUN_ROOT,
    DOWNSTREAM_ROOT,
    CACHE_ROOT,
    CHECKPOINT_ROOT,
    QUALITY_BASE_PRED_DIR,
    SYNTHETIC_PRED_LOWRES_ROOT,
    PREDICTED_MASK_ROOT,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# All three runs use the exact split created by the first run.
DOWNSTREAM_SPLIT_JSON = (
    LEGACY_DOWNSTREAM_ROOT
    / "downstream_training_split.json"
)


SEGMENTOR_BEST_CKPT = (
    CHECKPOINT_ROOT
    / "whole_tumour_segmentor_best.pt"
)

SEGMENTOR_LAST_CKPT = (
    CHECKPOINT_ROOT
    / "whole_tumour_segmentor_last.pt"
)

SEGMENTOR_DONE_JSON = (
    CHECKPOINT_ROOT
    / "whole_tumour_segmentor_done.json"
)

QUALITY_BEST_CKPT = (
    CHECKPOINT_ROOT
    / "whole_tumour_quality_predictor_best.pt"
)

QUALITY_LAST_CKPT = (
    CHECKPOINT_ROOT
    / "whole_tumour_quality_predictor_last.pt"
)

QUALITY_DONE_JSON = (
    CHECKPOINT_ROOT
    / "whole_tumour_quality_predictor_done.json"
)


FULL_SHAPE = (
    208,
    224,
    160
)

LOWRES_SHAPE = (
    104,
    112,
    80
)


SEG_TRAIN_N = 800
SEG_VAL_N = 100
QUALITY_TRAIN_N = 80
QUALITY_VAL_N = 20

SEG_BATCH_SIZE = 1
SEG_EPOCHS = 30
SEG_LEARNING_RATE = 2e-4
SEG_WEIGHT_DECAY = 1e-5
SEG_EARLY_STOPPING_PATIENCE = 7

QUALITY_BATCH_SIZE = 2
QUALITY_EPOCHS = 40
QUALITY_LEARNING_RATE = 1e-4
QUALITY_WEIGHT_DECAY = 1e-5
QUALITY_EARLY_STOPPING_PATIENCE = 8

SEGMENTATION_THRESHOLD = 0.5

NUM_WORKERS = 0

FORCE_REBUILD_REAL_CACHE = False
FORCE_REBUILD_SYNTHETIC_CACHE = False
FORCE_TRAIN_SEGMENTOR = False
FORCE_TRAIN_QUALITY_PREDICTOR = False
FORCE_SYNTHETIC_INFERENCE = False

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

AMP_ENABLED = (
    DEVICE.type
    == "cuda"
)


random.seed(
    TRAINING_SEED
)

np.random.seed(
    TRAINING_SEED
)

torch.manual_seed(
    TRAINING_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        TRAINING_SEED
    )


try:

    torch.set_float32_matmul_precision(
        "high"
    )

except Exception:

    pass


print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "Run label:",
    RUN_LABEL
)

print(
    "Training seed:",
    TRAINING_SEED
)

print(
    "Fixed split seed:",
    SPLIT_SEED
)

print(
    "Fixed perturbation seed:",
    PERTURBATION_SEED
)

print(
    "Fixed bootstrap seed:",
    BOOTSTRAP_SEED
)

print(
    "Run root:",
    RUN_ROOT
)

print(
    "Device:",
    DEVICE
)

if DEVICE.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print(
    "Full shape:",
    FULL_SHAPE
)

print(
    "Training shape:",
    LOWRES_SHAPE
)


Project root: /users/tdd540/Capstone_Project
Run label: v3
Training seed: 2028
Fixed split seed: 2026
Fixed perturbation seed: 2026
Fixed bootstrap seed: 2026
Run root: /users/tdd540/Capstone_Project/downstream_v3
Device: cuda
GPU: NVIDIA A40
Full shape: (208, 224, 160)
Training shape: (104, 112, 80)


In [3]:
required_paths = {
    "BraTS data directory":
        DATA_DIR,

    "data_split.json":
        DATA_SPLIT_JSON,

    "evaluation_subjects_200.json":
        EVALUATION_SUBJECTS_JSON,

    "condition-mask directory":
        CONDITION_MASK_DIR,
}


for name, path in (
    required_paths.items()
):

    if not path.exists():

        raise FileNotFoundError(
            f"Missing {name}: {path}"
        )

    print(
        f"Found {name}:",
        path
    )


with open(
    DATA_SPLIT_JSON,
    "r"
) as f:

    split_data = json.load(f)


with open(
    EVALUATION_SUBJECTS_JSON,
    "r"
) as f:

    evaluation_payload = json.load(f)


if isinstance(
    evaluation_payload,
    dict
):

    evaluation_subjects = list(
        evaluation_payload["subjects"]
    )

else:

    evaluation_subjects = list(
        evaluation_payload
    )


if len(
    evaluation_subjects
) != 200:

    raise RuntimeError(
        "The fixed evaluation cohort must contain "
        f"200 subjects, found {len(evaluation_subjects)}."
    )


for model_name, directory in (
    MODEL_DIRS.items()
):

    prefix = MODEL_PREFIXES[
        model_name
    ]

    files = sorted(
        directory.glob(
            f"{prefix}_*.nii.gz"
        )
    )

    if len(files) != 200:

        raise RuntimeError(
            f"{model_name} must contain 200 volumes; "
            f"found {len(files)}."
        )

    print(
        model_name,
        "synthetic volumes:",
        len(files)
    )


print(
    "Fixed evaluation subjects:",
    len(evaluation_subjects)
)


Found BraTS data directory: /users/tdd540/Capstone_Project/Data/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData
Found data_split.json: /users/tdd540/Capstone_Project/data_split.json
Found evaluation_subjects_200.json: /users/tdd540/Capstone_Project/evaluation_200/conditions/evaluation_subjects_200.json
Found condition-mask directory: /users/tdd540/Capstone_Project/evaluation_200/conditions/masks
ddpm_v5 synthetic volumes: 200
conditional_ddpm_v3 synthetic volumes: 200
conditional_ldm_v4 synthetic volumes: 200
Fixed evaluation subjects: 200


In [4]:
train_subjects = sorted(
    list(
        split_data["train"]
    )
)


expected_total = (
    SEG_TRAIN_N
    + SEG_VAL_N
    + QUALITY_TRAIN_N
    + QUALITY_VAL_N
)


if len(
    train_subjects
) < expected_total:

    raise RuntimeError(
        "The real generative-training split is too small "
        "for the requested downstream subdivisions."
    )


if not DOWNSTREAM_SPLIT_JSON.exists():

    raise FileNotFoundError(
        "The fixed downstream split from the first completed "
        f"run was not found: {DOWNSTREAM_SPLIT_JSON}"
    )


with open(
    DOWNSTREAM_SPLIT_JSON,
    "r"
) as f:

    downstream_split = json.load(f)


recorded_split_seed = int(
    downstream_split.get(
        "selection_seed",
        -1
    )
)


if recorded_split_seed != SPLIT_SEED:

    raise RuntimeError(
        "The saved downstream split has an unexpected seed: "
        f"{recorded_split_seed}; expected {SPLIT_SEED}."
    )


seg_train_subjects = list(
    downstream_split[
        "segmentor_train"
    ]
)

seg_val_subjects = list(
    downstream_split[
        "segmentor_validation"
    ]
)

quality_train_subjects = list(
    downstream_split[
        "quality_predictor_train"
    ]
)

quality_val_subjects = list(
    downstream_split[
        "quality_predictor_validation"
    ]
)


expected_sizes = {
    "segmentor_train":
        SEG_TRAIN_N,

    "segmentor_validation":
        SEG_VAL_N,

    "quality_predictor_train":
        QUALITY_TRAIN_N,

    "quality_predictor_validation":
        QUALITY_VAL_N,
}


actual_sizes = {
    "segmentor_train":
        len(
            seg_train_subjects
        ),

    "segmentor_validation":
        len(
            seg_val_subjects
        ),

    "quality_predictor_train":
        len(
            quality_train_subjects
        ),

    "quality_predictor_validation":
        len(
            quality_val_subjects
        ),
}


if actual_sizes != expected_sizes:

    raise RuntimeError(
        "The fixed downstream split has unexpected sizes: "
        f"{actual_sizes}"
    )


named_sets = {
    "segmentor_train":
        set(
            seg_train_subjects
        ),

    "segmentor_validation":
        set(
            seg_val_subjects
        ),

    "quality_predictor_train":
        set(
            quality_train_subjects
        ),

    "quality_predictor_validation":
        set(
            quality_val_subjects
        ),

    "synthetic_evaluation_conditions":
        set(
            evaluation_subjects
        ),
}


names = list(
    named_sets
)


for i in range(
    len(names)
):

    for j in range(
        i + 1,
        len(names)
    ):

        overlap = (
            named_sets[
                names[i]
            ]
            & named_sets[
                names[j]
            ]
        )

        if overlap:

            raise RuntimeError(
                "Downstream split leakage detected between "
                f"{names[i]} and {names[j]}: "
                f"{list(overlap)[:5]}"
            )


split_summary = pd.DataFrame(
    [
        {
            "Subset":
                "Segmentor training",

            "Subjects":
                len(
                    seg_train_subjects
                )
        },
        {
            "Subset":
                "Segmentor validation",

            "Subjects":
                len(
                    seg_val_subjects
                )
        },
        {
            "Subset":
                "Quality-predictor training",

            "Subjects":
                len(
                    quality_train_subjects
                )
        },
        {
            "Subset":
                "Quality-predictor validation",

            "Subjects":
                len(
                    quality_val_subjects
                )
        },
        {
            "Subset":
                "Synthetic evaluation conditions",

            "Subjects":
                len(
                    evaluation_subjects
                )
        },
    ]
)


display(
    split_summary
)


print(
    "Loaded fixed downstream split:",
    DOWNSTREAM_SPLIT_JSON
)

print(
    "Fixed split seed:",
    recorded_split_seed
)

print(
    "Current training seed:",
    TRAINING_SEED
)

print(
    "Downstream split validation passed."
)


,Subset,Subjects
0,Segmentor training,800
1,Segmentor validation,100
2,Quality-predictor training,80
3,Quality-predictor validation,20
4,Synthetic evaluation conditions,200


Loaded fixed downstream split: /users/tdd540/Capstone_Project/downstream_results/downstream_training_split.json
Fixed split seed: 2026
Current training seed: 2028
Downstream split validation passed.


In [5]:
def find_subject_file(
    subject: str,
    keyword: str
) -> Path:

    subject_dir = (
        DATA_DIR
        / subject
    )


    if not subject_dir.is_dir():

        raise FileNotFoundError(
            f"Subject directory missing: {subject_dir}"
        )


    candidates = [
        path
        for path in subject_dir.iterdir()
        if (
            keyword.lower()
            in path.name.lower()
            and (
                path.name.endswith(
                    ".nii"
                )
                or path.name.endswith(
                    ".nii.gz"
                )
            )
        )
    ]


    if len(candidates) != 1:

        raise RuntimeError(
            f"Expected one {keyword} file for {subject}; "
            f"found {[p.name for p in candidates]}"
        )


    return candidates[0]


def preprocess_t2f_01(
    image: np.ndarray
) -> np.ndarray:

    if image.shape != (
        240,
        240,
        155
    ):

        raise ValueError(
            f"Unexpected T2-FLAIR shape: {image.shape}"
        )


    image = image[
        16:224,
        8:232,
        :
    ]


    image = np.pad(
        image,
        (
            (0, 0),
            (0, 0),
            (2, 3)
        ),
        mode="constant",
        constant_values=0
    )


    foreground = (
        image > 0
    )


    if not np.any(
        foreground
    ):

        raise ValueError(
            "No foreground voxels found."
        )


    upper = np.percentile(
        image[
            foreground
        ],
        99.9
    )


    if upper <= 0:

        raise ValueError(
            "Invalid foreground percentile."
        )


    image = np.clip(
        image,
        0,
        upper
    )


    image = (
        image
        / upper
    )


    image[
        ~foreground
    ] = 0.0


    return image.astype(
        np.float32
    )


def preprocess_whole_tumour_mask(
    mask: np.ndarray
) -> np.ndarray:

    if mask.shape != (
        240,
        240,
        155
    ):

        raise ValueError(
            f"Unexpected segmentation shape: {mask.shape}"
        )


    mask = mask[
        16:224,
        8:232,
        :
    ]


    mask = np.pad(
        mask,
        (
            (0, 0),
            (0, 0),
            (2, 3)
        ),
        mode="constant",
        constant_values=0
    )


    # Whole tumour = union of every non-background BraTS label.
    mask = (
        mask > 0
    )


    return mask.astype(
        np.uint8
    )


def resize_image_np(
    image: np.ndarray,
    output_shape=LOWRES_SHAPE
) -> np.ndarray:

    tensor = (
        torch.from_numpy(
            image.astype(
                np.float32
            )
        )
        .unsqueeze(0)
        .unsqueeze(0)
    )


    resized = F.interpolate(
        tensor,
        size=output_shape,
        mode="trilinear",
        align_corners=False
    )


    return (
        resized[
            0,
            0
        ]
        .numpy()
        .astype(
            np.float32
        )
    )


def resize_mask_np(
    mask: np.ndarray,
    output_shape=LOWRES_SHAPE
) -> np.ndarray:

    tensor = (
        torch.from_numpy(
            mask.astype(
                np.float32
            )
        )
        .unsqueeze(0)
        .unsqueeze(0)
    )


    resized = F.interpolate(
        tensor,
        size=output_shape,
        mode="nearest"
    )


    return (
        resized[
            0,
            0
        ]
        .numpy()
        > 0.5
    ).astype(
        np.uint8
    )


def dice_np(
    prediction: np.ndarray,
    target: np.ndarray,
    eps: float = 1e-8
) -> float:

    prediction = (
        prediction > 0
    )

    target = (
        target > 0
    )


    prediction_sum = float(
        prediction.sum()
    )

    target_sum = float(
        target.sum()
    )


    if (
        prediction_sum == 0
        and target_sum == 0
    ):

        return 1.0


    intersection = float(
        np.logical_and(
            prediction,
            target
        ).sum()
    )


    return float(
        (
            2.0
            * intersection
            + eps
        )
        / (
            prediction_sum
            + target_sum
            + eps
        )
    )


def atomic_json_dump(
    payload,
    path: Path
):

    temporary_path = (
        path.with_suffix(
            path.suffix
            + ".tmp"
        )
    )


    with open(
        temporary_path,
        "w"
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            allow_nan=False
        )


    os.replace(
        temporary_path,
        path
    )


def atomic_torch_save(
    payload,
    path: Path
):

    temporary_path = (
        path.with_suffix(
            path.suffix
            + ".tmp"
        )
    )


    torch.save(
        payload,
        temporary_path
    )


    os.replace(
        temporary_path,
        path
    )


def atomic_csv_save(
    dataframe: pd.DataFrame,
    path: Path
):

    temporary_path = (
        path.with_suffix(
            path.suffix
            + ".tmp"
        )
    )


    dataframe.to_csv(
        temporary_path,
        index=False
    )


    os.replace(
        temporary_path,
        path
    )


In [6]:
all_real_subjects = []


for subject_list in [
    seg_train_subjects,
    seg_val_subjects,
    quality_train_subjects,
    quality_val_subjects,
    evaluation_subjects,
]:

    all_real_subjects.extend(
        subject_list
    )


all_real_subjects = list(
    dict.fromkeys(
        all_real_subjects
    )
)


missing_real_cache = [
    subject
    for subject in all_real_subjects
    if not (
        REAL_LOWRES_DIR
        / f"{subject}.npz"
    ).exists()
]


if missing_real_cache:

    raise FileNotFoundError(
        "The shared real low-resolution cache from the first "
        "completed run is incomplete. Missing examples: "
        f"{missing_real_cache[:5]}"
    )


print(
    "Verified shared real subjects:",
    len(
        all_real_subjects
    )
)

print(
    "Shared real cache directory:",
    REAL_LOWRES_DIR
)

print(
    "No shared cache files will be modified by this run."
)


Verified shared real subjects: 1200
Shared real cache directory: /users/tdd540/Capstone_Project/downstream_cache/real_lowres
No shared cache files will be modified by this run.


In [7]:
class LowResolutionBraTSDataset(
    Dataset
):

    def __init__(
        self,
        subjects,
        augment=False
    ):

        self.subjects = list(
            subjects
        )

        self.augment = bool(
            augment
        )


    def __len__(
        self
    ):

        return len(
            self.subjects
        )


    def __getitem__(
        self,
        index
    ):

        subject = self.subjects[
            index
        ]


        cache_path = (
            REAL_LOWRES_DIR
            / f"{subject}.npz"
        )


        with np.load(
            cache_path
        ) as cached:

            image = cached[
                "image"
            ].astype(
                np.float32
            )

            mask = cached[
                "mask"
            ].astype(
                np.float32
            )


        image = torch.from_numpy(
            image
        ).unsqueeze(0)


        mask = torch.from_numpy(
            mask
        ).unsqueeze(0)


        if self.augment:

            for spatial_dimension in [
                1,
                2,
                3
            ]:

                if random.random() < 0.5:

                    image = torch.flip(
                        image,
                        dims=[
                            spatial_dimension
                        ]
                    )

                    mask = torch.flip(
                        mask,
                        dims=[
                            spatial_dimension
                        ]
                    )


            intensity_scale = random.uniform(
                0.90,
                1.10
            )

            intensity_shift = random.uniform(
                -0.05,
                0.05
            )


            image = (
                image
                * intensity_scale
                + intensity_shift
            )


            if random.random() < 0.35:

                image = (
                    image
                    + torch.randn_like(
                        image
                    )
                    * random.uniform(
                        0.0,
                        0.025
                    )
                )


            image = torch.clamp(
                image,
                0.0,
                1.0
            )


        return {
            "image":
                image.contiguous(),

            "mask":
                mask.contiguous(),

            "subject":
                subject,
        }


In [8]:
class ConvBlock3D(
    nn.Module
):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()


        self.block = nn.Sequential(
            nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True
            ),
            nn.LeakyReLU(
                negative_slope=0.1,
                inplace=True
            ),
            nn.Conv3d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True
            ),
            nn.LeakyReLU(
                negative_slope=0.1,
                inplace=True
            ),
        )


    def forward(
        self,
        x
    ):

        return self.block(
            x
        )


class WholeTumourUNet3D(
    nn.Module
):

    def __init__(
        self,
        base_channels=16
    ):

        super().__init__()


        c1 = base_channels
        c2 = base_channels * 2
        c3 = base_channels * 4
        c4 = base_channels * 8


        self.encoder1 = ConvBlock3D(
            1,
            c1
        )

        self.encoder2 = ConvBlock3D(
            c1,
            c2
        )

        self.encoder3 = ConvBlock3D(
            c2,
            c3
        )

        self.bottleneck = ConvBlock3D(
            c3,
            c4
        )


        self.pool = nn.MaxPool3d(
            kernel_size=2,
            stride=2
        )


        self.up3 = nn.ConvTranspose3d(
            c4,
            c3,
            kernel_size=2,
            stride=2
        )

        self.decoder3 = ConvBlock3D(
            c3 + c3,
            c3
        )


        self.up2 = nn.ConvTranspose3d(
            c3,
            c2,
            kernel_size=2,
            stride=2
        )

        self.decoder2 = ConvBlock3D(
            c2 + c2,
            c2
        )


        self.up1 = nn.ConvTranspose3d(
            c2,
            c1,
            kernel_size=2,
            stride=2
        )

        self.decoder1 = ConvBlock3D(
            c1 + c1,
            c1
        )


        self.output = nn.Conv3d(
            c1,
            1,
            kernel_size=1
        )


    def forward(
        self,
        x
    ):

        e1 = self.encoder1(
            x
        )

        e2 = self.encoder2(
            self.pool(
                e1
            )
        )

        e3 = self.encoder3(
            self.pool(
                e2
            )
        )

        bottleneck = self.bottleneck(
            self.pool(
                e3
            )
        )


        d3 = self.up3(
            bottleneck
        )

        d3 = self.decoder3(
            torch.cat(
                [
                    d3,
                    e3
                ],
                dim=1
            )
        )


        d2 = self.up2(
            d3
        )

        d2 = self.decoder2(
            torch.cat(
                [
                    d2,
                    e2
                ],
                dim=1
            )
        )


        d1 = self.up1(
            d2
        )

        d1 = self.decoder1(
            torch.cat(
                [
                    d1,
                    e1
                ],
                dim=1
            )
        )


        return self.output(
            d1
        )


def soft_dice_loss(
    logits,
    targets,
    eps=1e-6
):

    probabilities = torch.sigmoid(
        logits
    )


    dimensions = tuple(
        range(
            2,
            logits.ndim
        )
    )


    intersection = torch.sum(
        probabilities
        * targets,
        dim=dimensions
    )


    denominator = (
        torch.sum(
            probabilities,
            dim=dimensions
        )
        + torch.sum(
            targets,
            dim=dimensions
        )
    )


    dice = (
        2.0
        * intersection
        + eps
    ) / (
        denominator
        + eps
    )


    return (
        1.0
        - dice.mean()
    )


def whole_tumour_loss(
    logits,
    targets
):

    bce = (
        F.binary_cross_entropy_with_logits(
            logits,
            targets
        )
    )


    dice = soft_dice_loss(
        logits,
        targets
    )


    return (
        0.5
        * bce
        + 0.5
        * dice
    )


@torch.no_grad()
def batch_binary_dice(
    logits,
    targets,
    threshold=SEGMENTATION_THRESHOLD,
    eps=1e-6
):

    predictions = (
        torch.sigmoid(
            logits
        )
        >= threshold
    )


    targets = (
        targets > 0.5
    )


    dimensions = tuple(
        range(
            2,
            logits.ndim
        )
    )


    intersection = torch.sum(
        predictions
        & targets,
        dim=dimensions
    ).float()


    denominator = (
        torch.sum(
            predictions,
            dim=dimensions
        ).float()
        + torch.sum(
            targets,
            dim=dimensions
        ).float()
    )


    dice = (
        2.0
        * intersection
        + eps
    ) / (
        denominator
        + eps
    )


    return dice


In [9]:
seg_train_dataset = (
    LowResolutionBraTSDataset(
        seg_train_subjects,
        augment=True
    )
)


seg_val_dataset = (
    LowResolutionBraTSDataset(
        seg_val_subjects,
        augment=False
    )
)


seg_train_loader = DataLoader(
    seg_train_dataset,
    batch_size=SEG_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(
        DEVICE.type
        == "cuda"
    ),
)


seg_val_loader = DataLoader(
    seg_val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(
        DEVICE.type
        == "cuda"
    ),
)


segmentor = (
    WholeTumourUNet3D(
        base_channels=16
    )
    .to(
        DEVICE
    )
)


smoke_batch = next(
    iter(
        seg_train_loader
    )
)


smoke_image = (
    smoke_batch[
        "image"
    ]
    .to(
        DEVICE
    )
)


with torch.inference_mode():

    smoke_logits = segmentor(
        smoke_image
    )


print(
    "Segmentor input:",
    tuple(
        smoke_image.shape
    )
)

print(
    "Segmentor output:",
    tuple(
        smoke_logits.shape
    )
)


assert (
    smoke_logits.shape
    == smoke_batch[
        "mask"
    ].shape
)


print(
    "Whole-tumour segmentor smoke test: PASS"
)


del smoke_batch
del smoke_image
del smoke_logits

if torch.cuda.is_available():

    torch.cuda.empty_cache()


Segmentor input: (1, 1, 104, 112, 80)
Segmentor output: (1, 1, 104, 112, 80)
Whole-tumour segmentor smoke test: PASS


In [10]:
def evaluate_segmentor_loader(
    model,
    loader
):

    model.eval()


    losses = []
    dices = []


    with torch.inference_mode():

        for batch in loader:

            images = (
                batch[
                    "image"
                ]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )

            masks = (
                batch[
                    "mask"
                ]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED
            ):

                logits = model(
                    images
                )

                loss = whole_tumour_loss(
                    logits,
                    masks
                )


            batch_dice = batch_binary_dice(
                logits,
                masks
            )


            losses.append(
                float(
                    loss.item()
                )
            )

            dices.extend(
                batch_dice
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )


    return {
        "loss":
            float(
                np.mean(
                    losses
                )
            ),

        "dice":
            float(
                np.mean(
                    dices
                )
            ),
    }


optimizer = torch.optim.AdamW(
    segmentor.parameters(),
    lr=SEG_LEARNING_RATE,
    weight_decay=SEG_WEIGHT_DECAY
)


scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)


start_epoch = 1
best_val_dice = -np.inf
epochs_without_improvement = 0
segmentor_history = []


training_required = (
    FORCE_TRAIN_SEGMENTOR
    or not SEGMENTOR_DONE_JSON.exists()
)


if training_required:

    if (
        SEGMENTOR_LAST_CKPT.exists()
        and not FORCE_TRAIN_SEGMENTOR
    ):

        checkpoint = torch.load(
            SEGMENTOR_LAST_CKPT,
            map_location=DEVICE
        )


        segmentor.load_state_dict(
            checkpoint[
                "model_state_dict"
            ]
        )

        optimizer.load_state_dict(
            checkpoint[
                "optimizer_state_dict"
            ]
        )


        start_epoch = (
            int(
                checkpoint[
                    "epoch"
                ]
            )
            + 1
        )

        best_val_dice = float(
            checkpoint[
                "best_val_dice"
            ]
        )

        epochs_without_improvement = int(
            checkpoint.get(
                "epochs_without_improvement",
                0
            )
        )

        segmentor_history = list(
            checkpoint.get(
                "history",
                []
            )
        )


        print(
            "Resuming segmentor from epoch:",
            start_epoch
        )


    for epoch in range(
        start_epoch,
        SEG_EPOCHS + 1
    ):

        segmentor.train()


        epoch_losses = []
        epoch_dices = []


        progress = tqdm(
            seg_train_loader,
            desc=(
                f"Segmentor epoch "
                f"{epoch}/{SEG_EPOCHS}"
            )
        )


        for batch in progress:

            images = (
                batch[
                    "image"
                ]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )

            masks = (
                batch[
                    "mask"
                ]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED
            ):

                logits = segmentor(
                    images
                )

                loss = whole_tumour_loss(
                    logits,
                    masks
                )


            scaler.scale(
                loss
            ).backward()


            scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                segmentor.parameters(),
                max_norm=5.0
            )


            scaler.step(
                optimizer
            )

            scaler.update()


            batch_dice = batch_binary_dice(
                logits.detach(),
                masks
            )


            epoch_losses.append(
                float(
                    loss.item()
                )
            )

            epoch_dices.extend(
                batch_dice
                .cpu()
                .numpy()
                .tolist()
            )


            progress.set_postfix({
                "loss":
                    f"{np.mean(epoch_losses):.4f}",

                "dice":
                    f"{np.mean(epoch_dices):.4f}",
            })


        validation = evaluate_segmentor_loader(
            segmentor,
            seg_val_loader
        )


        scheduler.step(
            validation[
                "dice"
            ]
        )


        row = {
            "epoch":
                epoch,

            "train_loss":
                float(
                    np.mean(
                        epoch_losses
                    )
                ),

            "train_dice":
                float(
                    np.mean(
                        epoch_dices
                    )
                ),

            "val_loss":
                validation[
                    "loss"
                ],

            "val_dice":
                validation[
                    "dice"
                ],

            "learning_rate":
                float(
                    optimizer.param_groups[
                        0
                    ][
                        "lr"
                    ]
                ),
        }


        segmentor_history.append(
            row
        )


        print(
            row
        )


        if (
            validation[
                "dice"
            ]
            > best_val_dice
        ):

            best_val_dice = (
                validation[
                    "dice"
                ]
            )

            epochs_without_improvement = 0


            atomic_torch_save(
                {
                    "epoch":
                        epoch,

                    "model_state_dict":
                        segmentor.state_dict(),

                    "best_val_dice":
                        best_val_dice,

                    "config": {
                        "lowres_shape":
                            LOWRES_SHAPE,

                        "threshold":
                            SEGMENTATION_THRESHOLD,

                        "base_channels":
                            16,
                    },
                },
                SEGMENTOR_BEST_CKPT
            )

        else:

            epochs_without_improvement += 1


        atomic_torch_save(
            {
                "epoch":
                    epoch,

                "model_state_dict":
                    segmentor.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "best_val_dice":
                    best_val_dice,

                "epochs_without_improvement":
                    epochs_without_improvement,

                "history":
                    segmentor_history,
            },
            SEGMENTOR_LAST_CKPT
        )


        atomic_csv_save(
            pd.DataFrame(
                segmentor_history
            ),
            DOWNSTREAM_ROOT
            / "segmentor_training_history.csv"
        )


        if (
            epochs_without_improvement
            >= SEG_EARLY_STOPPING_PATIENCE
        ):

            print(
                "Segmentor early stopping triggered."
            )

            break


    atomic_json_dump(
        {
            "completed":
                True,

            "best_validation_dice":
                float(
                    best_val_dice
                ),

            "best_checkpoint":
                str(
                    SEGMENTOR_BEST_CKPT
                ),

            "last_epoch":
                int(
                    segmentor_history[
                        -1
                    ][
                        "epoch"
                    ]
                ),
        },
        SEGMENTOR_DONE_JSON
    )

else:

    print(
        "Segmentor training already completed -> skipped"
    )


if not SEGMENTOR_BEST_CKPT.exists():

    raise FileNotFoundError(
        "Best segmentor checkpoint was not created."
    )


best_segmentor_checkpoint = torch.load(
    SEGMENTOR_BEST_CKPT,
    map_location=DEVICE
)


segmentor.load_state_dict(
    best_segmentor_checkpoint[
        "model_state_dict"
    ]
)


segmentor.eval()


print(
    "Loaded best segmentor validation Dice:",
    best_segmentor_checkpoint[
        "best_val_dice"
    ]
)


Segmentor epoch 1/30:   0%|          | 0/800 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.6081388412043452, 'train_dice': 0.658727713029366, 'val_loss': 0.521875302195549, 'val_dice': 0.6697082566097379, 'learning_rate': 0.0002}


Segmentor epoch 2/30:   0%|          | 0/800 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 0.43073252592235806, 'train_dice': 0.7106723200286492, 'val_loss': 0.3480058251321316, 'val_dice': 0.702877645753324, 'learning_rate': 0.0002}


Segmentor epoch 3/30:   0%|          | 0/800 [00:00<?, ?it/s]

...

{'epoch': 28, 'train_loss': 0.060450625461526214, 'train_dice': 0.8940589708089829, 'val_loss': 0.0725517818890512, 'val_dice': 0.8696552452730745, 'learning_rate': 1.25e-05}


Segmentor epoch 29/30:   0%|          | 0/800 [00:00<?, ?it/s]

{'epoch': 29, 'train_loss': 0.05975953961024061, 'train_dice': 0.8951988431454274, 'val_loss': 0.07168802678585053, 'val_dice': 0.8713316911744684, 'learning_rate': 1.25e-05}


Segmentor epoch 30/30:   0%|          | 0/800 [00:00<?, ?it/s]

{'epoch': 30, 'train_loss': 0.059056546224746855, 'train_dice': 0.896705320328474, 'val_loss': 0.07171189069747924, 'val_dice': 0.8712519800948709, 'learning_rate': 1.25e-05}
Loaded best segmentor validation Dice: 0.8719718072104067


In [11]:
@torch.no_grad()
def predict_lowres_whole_tumour(
    model,
    image_np
):

    image_tensor = (
        torch.from_numpy(
            image_np.astype(
                np.float32
            )
        )
        .unsqueeze(0)
        .unsqueeze(0)
        .to(
            DEVICE
        )
    )


    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=AMP_ENABLED
    ):

        logits = model(
            image_tensor
        )


    probability = (
        torch.sigmoid(
            logits
        )[
            0,
            0
        ]
        .float()
        .cpu()
        .numpy()
    )


    prediction = (
        probability
        >= SEGMENTATION_THRESHOLD
    ).astype(
        np.uint8
    )


    return (
        probability.astype(
            np.float32
        ),
        prediction
    )


def evaluate_real_subjects(
    subjects,
    subset_name
):

    rows = []


    for subject in tqdm(
        subjects,
        desc=f"Real WT evaluation: {subset_name}"
    ):

        cache_path = (
            REAL_LOWRES_DIR
            / f"{subject}.npz"
        )


        with np.load(
            cache_path
        ) as cached:

            image = cached[
                "image"
            ].astype(
                np.float32
            )

            ground_truth = cached[
                "mask"
            ].astype(
                np.uint8
            )


        probability, prediction = (
            predict_lowres_whole_tumour(
                segmentor,
                image
            )
        )


        rows.append({
            "subject":
                subject,

            "subset":
                subset_name,

            "actual_dice":
                dice_np(
                    prediction,
                    ground_truth
                ),

            "predicted_tumour_voxels":
                int(
                    prediction.sum()
                ),

            "ground_truth_tumour_voxels":
                int(
                    ground_truth.sum()
                ),

            "mean_tumour_probability":
                float(
                    probability[
                        prediction > 0
                    ].mean()
                )
                if prediction.any()
                else 0.0,
        })


    return pd.DataFrame(
        rows
    )


seg_val_performance = (
    evaluate_real_subjects(
        seg_val_subjects,
        "segmentor_validation"
    )
)


heldout_real_performance = (
    evaluate_real_subjects(
        evaluation_subjects,
        "fixed_heldout_200"
    )
)


segmentor_real_performance = pd.concat(
    [
        seg_val_performance,
        heldout_real_performance
    ],
    ignore_index=True
)


atomic_csv_save(
    segmentor_real_performance,
    DOWNSTREAM_ROOT
    / "real_whole_tumour_segmentor_performance.csv"
)


display(
    segmentor_real_performance
    .groupby(
        "subset"
    )[
        "actual_dice"
    ]
    .agg(
        [
            "mean",
            "std",
            "median",
            "min",
            "max"
        ]
    )
)


print(
    "Real segmentor performance saved."
)


Real WT evaluation: segmentor_validation:   0%|          | 0/100 [00:00<?, ?it/s]

Real WT evaluation: fixed_heldout_200:   0%|          | 0/200 [00:00<?, ?it/s]

,mean,std,median,min,max
subset,,,,,
fixed_heldout_200,0.886289,0.106484,0.919627,2.032466e-01,0.968953
segmentor_validation,0.871972,0.131174,0.916179,2.857143e-11,0.970644


Real segmentor performance saved.


In [12]:
quality_all_subjects = (
    quality_train_subjects
    + quality_val_subjects
)


for subject in tqdm(
    quality_all_subjects,
    desc="Base WT predictions for quality model"
):

    output_path = (
        QUALITY_BASE_PRED_DIR
        / f"{subject}.npz"
    )


    if output_path.exists():

        continue


    with np.load(
        REAL_LOWRES_DIR
        / f"{subject}.npz"
    ) as cached:

        image = cached[
            "image"
        ].astype(
            np.float32
        )


    probability, prediction = (
        predict_lowres_whole_tumour(
            segmentor,
            image
        )
    )


    np.savez(
        output_path,
        probability=probability.astype(
            np.float16
        ),
        prediction=prediction.astype(
            np.uint8
        ),
    )


missing_base_predictions = [
    subject
    for subject in quality_all_subjects
    if not (
        QUALITY_BASE_PRED_DIR
        / f"{subject}.npz"
    ).exists()
]


if missing_base_predictions:

    raise RuntimeError(
        "Quality-model base predictions are incomplete."
    )


print(
    "Quality-model base predictions:",
    len(
        quality_all_subjects
    )
)


# Release GPU memory before training the quality regressor.
segmentor.to(
    "cpu"
)

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


Base WT predictions for quality model:   0%|          | 0/100 [00:00<?, ?it/s]

Quality-model base predictions: 100


In [13]:
MORPHOLOGY_STRUCTURE = (
    generate_binary_structure(
        3,
        1
    )
)


PERTURBATION_KINDS = [
    "baseline",
    "erode_1",
    "erode_2",
    "erode_3",
    "dilate_1",
    "dilate_2",
    "dilate_3",
    "shift_small",
    "shift_large",
    "cutout_small",
    "cutout_large",
    "add_blob",
    "boundary_noise",
]


def ensure_nonempty_mask(
    mask,
    fallback
):

    mask = (
        mask > 0
    )


    if mask.any():

        return mask.astype(
            np.uint8
        )


    return (
        fallback > 0
    ).astype(
        np.uint8
    )


def zero_fill_shift(
    mask,
    shifts
):

    shifted = ndi_shift(
        mask.astype(
            np.uint8
        ),
        shift=shifts,
        order=0,
        mode="constant",
        cval=0,
        prefilter=False
    )


    return (
        shifted > 0
    ).astype(
        np.uint8
    )


def add_random_blob(
    mask,
    rng
):

    output = (
        mask > 0
    ).copy()


    shape = np.asarray(
        output.shape
    )


    radius = int(
        rng.integers(
            3,
            8
        )
    )


    centre = np.asarray(
        [
            rng.integers(
                radius,
                max(
                    radius + 1,
                    shape[axis]
                    - radius
                )
            )
            for axis in range(3)
        ],
        dtype=int
    )


    lower = np.maximum(
        centre - radius,
        0
    )

    upper = np.minimum(
        centre + radius + 1,
        shape
    )


    local_shape = (
        upper
        - lower
    )


    coordinates = np.ogrid[
        tuple(
            slice(
                0,
                int(length)
            )
            for length in local_shape
        )
    ]


    squared_distance = np.zeros(
        tuple(
            local_shape
        ),
        dtype=np.float32
    )


    local_centre = (
        centre
        - lower
    )


    for axis, grid in enumerate(
        coordinates
    ):

        squared_distance += (
            grid
            - local_centre[
                axis
            ]
        ) ** 2


    sphere = (
        squared_distance
        <= radius ** 2
    )


    target_slices = tuple(
        slice(
            int(lower[axis]),
            int(upper[axis])
        )
        for axis in range(3)
    )


    output[
        target_slices
    ] |= sphere


    return output.astype(
        np.uint8
    )


def apply_mask_perturbation(
    base_mask,
    ground_truth,
    kind,
    seed
):

    rng = np.random.default_rng(
        seed
    )


    base_mask = (
        base_mask > 0
    ).astype(
        np.uint8
    )

    ground_truth = (
        ground_truth > 0
    ).astype(
        np.uint8
    )


    if not base_mask.any():

        base_mask = ground_truth.copy()


    if kind == "baseline":

        output = base_mask


    elif kind.startswith(
        "erode_"
    ):

        iterations = int(
            kind.split(
                "_"
            )[
                1
            ]
        )


        output = binary_erosion(
            base_mask,
            structure=MORPHOLOGY_STRUCTURE,
            iterations=iterations
        ).astype(
            np.uint8
        )


    elif kind.startswith(
        "dilate_"
    ):

        iterations = int(
            kind.split(
                "_"
            )[
                1
            ]
        )


        output = binary_dilation(
            base_mask,
            structure=MORPHOLOGY_STRUCTURE,
            iterations=iterations
        ).astype(
            np.uint8
        )


    elif kind == "shift_small":

        shifts = tuple(
            int(value)
            for value in rng.integers(
                -2,
                3,
                size=3
            )
        )


        output = zero_fill_shift(
            base_mask,
            shifts
        )


    elif kind == "shift_large":

        shifts = tuple(
            int(value)
            for value in rng.integers(
                -5,
                6,
                size=3
            )
        )


        output = zero_fill_shift(
            base_mask,
            shifts
        )


    elif kind.startswith(
        "cutout_"
    ):

        output = base_mask.copy()


        tumour_coordinates = np.argwhere(
            output > 0
        )


        if len(
            tumour_coordinates
        ) > 0:

            minimum = tumour_coordinates.min(
                axis=0
            )

            maximum = tumour_coordinates.max(
                axis=0
            )


            centre = np.asarray(
                [
                    rng.integers(
                        int(
                            minimum[
                                axis
                            ]
                        ),
                        int(
                            maximum[
                                axis
                            ]
                        )
                        + 1
                    )
                    for axis in range(3)
                ]
            )


            fraction = (
                0.25
                if kind
                == "cutout_small"
                else 0.45
            )


            extent = np.maximum(
                (
                    (
                        maximum
                        - minimum
                        + 1
                    )
                    * fraction
                ).astype(
                    int
                ),
                2
            )


            lower = np.maximum(
                centre
                - extent
                // 2,
                0
            )

            upper = np.minimum(
                lower
                + extent,
                np.asarray(
                    output.shape
                )
            )


            output[
                int(lower[0]):
                int(upper[0]),

                int(lower[1]):
                int(upper[1]),

                int(lower[2]):
                int(upper[2])
            ] = 0


    elif kind == "add_blob":

        output = add_random_blob(
            base_mask,
            rng
        )


    elif kind == "boundary_noise":

        dilated = binary_dilation(
            base_mask,
            structure=MORPHOLOGY_STRUCTURE,
            iterations=2
        )


        eroded = binary_erosion(
            base_mask,
            structure=MORPHOLOGY_STRUCTURE,
            iterations=2
        )


        boundary = np.logical_xor(
            dilated,
            eroded
        )


        output = base_mask.astype(
            bool
        )


        boundary_coordinates = np.argwhere(
            boundary
        )


        if len(
            boundary_coordinates
        ) > 0:

            selected_n = max(
                1,
                int(
                    len(
                        boundary_coordinates
                    )
                    * 0.25
                )
            )


            selected_indices = rng.choice(
                len(
                    boundary_coordinates
                ),
                size=selected_n,
                replace=False
            )


            selected = boundary_coordinates[
                selected_indices
            ]


            output[
                selected[
                    :,
                    0
                ],
                selected[
                    :,
                    1
                ],
                selected[
                    :,
                    2
                ]
            ] = ~output[
                selected[
                    :,
                    0
                ],
                selected[
                    :,
                    1
                ],
                selected[
                    :,
                    2
                ]
            ]


        output = binary_closing(
            output,
            structure=MORPHOLOGY_STRUCTURE,
            iterations=1
        )


        output = binary_opening(
            output,
            structure=MORPHOLOGY_STRUCTURE,
            iterations=1
        ).astype(
            np.uint8
        )


    else:

        raise ValueError(
            f"Unknown perturbation: {kind}"
        )


    output = ensure_nonempty_mask(
        output,
        fallback=base_mask
    )


    return output


class WholeTumourQualityDataset(
    Dataset
):

    def __init__(
        self,
        subjects,
        augment=False
    ):

        self.subjects = list(
            subjects
        )

        self.augment = bool(
            augment
        )


        self.samples = [
            (
                subject,
                perturbation
            )
            for subject in self.subjects
            for perturbation in PERTURBATION_KINDS
        ]


    def __len__(
        self
    ):

        return len(
            self.samples
        )


    def __getitem__(
        self,
        index
    ):

        subject, perturbation = (
            self.samples[
                index
            ]
        )


        with np.load(
            REAL_LOWRES_DIR
            / f"{subject}.npz"
        ) as cached:

            image = cached[
                "image"
            ].astype(
                np.float32
            )

            ground_truth = cached[
                "mask"
            ].astype(
                np.uint8
            )


        with np.load(
            QUALITY_BASE_PRED_DIR
            / f"{subject}.npz"
        ) as cached_prediction:

            base_prediction = (
                cached_prediction[
                    "prediction"
                ]
                .astype(
                    np.uint8
                )
            )


        perturbation_index = (
            PERTURBATION_KINDS.index(
                perturbation
            )
        )


        subject_seed = (
            PERTURBATION_SEED
            + sum(
                ord(
                    character
                )
                for character in subject
            )
            * 100
            + perturbation_index
        )


        perturbed_mask = (
            apply_mask_perturbation(
                base_prediction,
                ground_truth,
                perturbation,
                subject_seed
            )
        )


        target_dice = dice_np(
            perturbed_mask,
            ground_truth
        )


        image_tensor = torch.from_numpy(
            image
        ).float()


        mask_tensor = torch.from_numpy(
            perturbed_mask.astype(
                np.float32
            )
        )


        if self.augment:

            for spatial_dimension in [
                0,
                1,
                2
            ]:

                if random.random() < 0.5:

                    image_tensor = torch.flip(
                        image_tensor,
                        dims=[
                            spatial_dimension
                        ]
                    )

                    mask_tensor = torch.flip(
                        mask_tensor,
                        dims=[
                            spatial_dimension
                        ]
                    )


            image_tensor = torch.clamp(
                image_tensor
                * random.uniform(
                    0.95,
                    1.05
                )
                + random.uniform(
                    -0.025,
                    0.025
                ),
                0.0,
                1.0
            )


        paired_input = torch.stack(
            [
                image_tensor,
                mask_tensor
            ],
            dim=0
        )


        return {
            "input":
                paired_input.contiguous(),

            "target":
                torch.tensor(
                    target_dice,
                    dtype=torch.float32
                ),

            "subject":
                subject,

            "perturbation":
                perturbation,
        }


In [14]:
def group_norm(
    channels
):

    groups = min(
        8,
        channels
    )


    while (
        channels
        % groups
        != 0
    ):

        groups -= 1


    return nn.GroupNorm(
        groups,
        channels
    )


class ResidualBlock3D(
    nn.Module
):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1
    ):

        super().__init__()


        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )

        self.norm1 = group_norm(
            out_channels
        )


        self.conv2 = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.norm2 = group_norm(
            out_channels
        )


        if (
            stride != 1
            or in_channels
            != out_channels
        ):

            self.skip = nn.Sequential(
                nn.Conv3d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                group_norm(
                    out_channels
                ),
            )

        else:

            self.skip = nn.Identity()


    def forward(
        self,
        x
    ):

        residual = self.skip(
            x
        )


        x = F.silu(
            self.norm1(
                self.conv1(
                    x
                )
            )
        )


        x = self.norm2(
            self.conv2(
                x
            )
        )


        x = F.silu(
            x
            + residual
        )


        return x


class WholeTumourQualityRegressor3D(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()


        self.stem = nn.Sequential(
            nn.Conv3d(
                2,
                16,
                kernel_size=5,
                stride=2,
                padding=2,
                bias=False
            ),
            group_norm(
                16
            ),
            nn.SiLU(),
        )


        self.layer1 = nn.Sequential(
            ResidualBlock3D(
                16,
                16
            ),
            ResidualBlock3D(
                16,
                16
            ),
        )


        self.layer2 = nn.Sequential(
            ResidualBlock3D(
                16,
                32,
                stride=2
            ),
            ResidualBlock3D(
                32,
                32
            ),
        )


        self.layer3 = nn.Sequential(
            ResidualBlock3D(
                32,
                64,
                stride=2
            ),
            ResidualBlock3D(
                64,
                64
            ),
        )


        self.layer4 = nn.Sequential(
            ResidualBlock3D(
                64,
                128,
                stride=2
            ),
            ResidualBlock3D(
                128,
                128
            ),
        )


        self.pool = nn.AdaptiveAvgPool3d(
            1
        )


        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(
                128,
                64
            ),
            nn.SiLU(),
            nn.Dropout(
                p=0.20
            ),
            nn.Linear(
                64,
                1
            ),
            nn.Sigmoid(),
        )


    def forward(
        self,
        x
    ):

        x = self.stem(
            x
        )

        x = self.layer1(
            x
        )

        x = self.layer2(
            x
        )

        x = self.layer3(
            x
        )

        x = self.layer4(
            x
        )

        x = self.pool(
            x
        )


        return self.regressor(
            x
        ).squeeze(
            1
        )


In [15]:
quality_train_dataset = (
    WholeTumourQualityDataset(
        quality_train_subjects,
        augment=True
    )
)


quality_val_dataset = (
    WholeTumourQualityDataset(
        quality_val_subjects,
        augment=False
    )
)


quality_train_loader = DataLoader(
    quality_train_dataset,
    batch_size=QUALITY_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(
        DEVICE.type
        == "cuda"
    ),
)


quality_val_loader = DataLoader(
    quality_val_dataset,
    batch_size=QUALITY_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(
        DEVICE.type
        == "cuda"
    ),
)


quality_predictor = (
    WholeTumourQualityRegressor3D()
    .to(
        DEVICE
    )
)


quality_smoke_batch = next(
    iter(
        quality_train_loader
    )
)


quality_smoke_input = (
    quality_smoke_batch[
        "input"
    ]
    .to(
        DEVICE
    )
)


with torch.inference_mode():

    quality_smoke_output = (
        quality_predictor(
            quality_smoke_input
        )
    )


print(
    "Quality input:",
    tuple(
        quality_smoke_input.shape
    )
)

print(
    "Quality output:",
    tuple(
        quality_smoke_output.shape
    )
)

print(
    "Quality output range:",
    float(
        quality_smoke_output.min()
    ),
    float(
        quality_smoke_output.max()
    )
)


assert (
    quality_smoke_output.ndim
    == 1
)

assert torch.isfinite(
    quality_smoke_output
).all()


print(
    "WT quality-regressor smoke test: PASS"
)


del quality_smoke_batch
del quality_smoke_input
del quality_smoke_output

if torch.cuda.is_available():

    torch.cuda.empty_cache()


Quality input: (2, 2, 104, 112, 80)
Quality output: (2,)
Quality output range: 0.4910042881965637 0.520377516746521


WT quality-regressor smoke test: PASS


In [16]:
def quality_validation_metrics(
    model,
    loader
):

    model.eval()


    predictions = []
    targets = []
    losses = []


    with torch.inference_mode():

        for batch in loader:

            inputs = (
                batch[
                    "input"
                ]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )

            target = (
                batch[
                    "target"
                ]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED
            ):

                prediction = model(
                    inputs
                )

                loss = F.mse_loss(
                    prediction,
                    target
                )


            predictions.extend(
                prediction
                .float()
                .cpu()
                .numpy()
                .tolist()
            )

            targets.extend(
                target
                .float()
                .cpu()
                .numpy()
                .tolist()
            )

            losses.append(
                float(
                    loss.item()
                )
            )


    predictions = np.asarray(
        predictions,
        dtype=np.float64
    )

    targets = np.asarray(
        targets,
        dtype=np.float64
    )


    mae = float(
        np.mean(
            np.abs(
                predictions
                - targets
            )
        )
    )


    rmse = float(
        np.sqrt(
            np.mean(
                (
                    predictions
                    - targets
                ) ** 2
            )
        )
    )


    if (
        np.std(
            predictions
        )
        > 0
        and np.std(
            targets
        )
        > 0
    ):

        pearson_value = float(
            pearsonr(
                predictions,
                targets
            ).statistic
        )

    else:

        pearson_value = np.nan


    return {
        "loss":
            float(
                np.mean(
                    losses
                )
            ),

        "mae":
            mae,

        "rmse":
            rmse,

        "pearson":
            pearson_value,

        "predictions":
            predictions,

        "targets":
            targets,
    }


quality_optimizer = torch.optim.AdamW(
    quality_predictor.parameters(),
    lr=QUALITY_LEARNING_RATE,
    weight_decay=QUALITY_WEIGHT_DECAY
)


quality_scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        quality_optimizer,
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
)


quality_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)


quality_start_epoch = 1
best_quality_mae = np.inf
quality_epochs_without_improvement = 0
quality_history = []


quality_training_required = (
    FORCE_TRAIN_QUALITY_PREDICTOR
    or not QUALITY_DONE_JSON.exists()
)


if quality_training_required:

    if (
        QUALITY_LAST_CKPT.exists()
        and not FORCE_TRAIN_QUALITY_PREDICTOR
    ):

        checkpoint = torch.load(
            QUALITY_LAST_CKPT,
            map_location=DEVICE
        )


        quality_predictor.load_state_dict(
            checkpoint[
                "model_state_dict"
            ]
        )

        quality_optimizer.load_state_dict(
            checkpoint[
                "optimizer_state_dict"
            ]
        )


        quality_start_epoch = (
            int(
                checkpoint[
                    "epoch"
                ]
            )
            + 1
        )

        best_quality_mae = float(
            checkpoint[
                "best_val_mae"
            ]
        )

        quality_epochs_without_improvement = int(
            checkpoint.get(
                "epochs_without_improvement",
                0
            )
        )

        quality_history = list(
            checkpoint.get(
                "history",
                []
            )
        )


        print(
            "Resuming quality predictor from epoch:",
            quality_start_epoch
        )


    for epoch in range(
        quality_start_epoch,
        QUALITY_EPOCHS + 1
    ):

        quality_predictor.train()


        epoch_losses = []


        progress = tqdm(
            quality_train_loader,
            desc=(
                f"Quality epoch "
                f"{epoch}/{QUALITY_EPOCHS}"
            )
        )


        for batch in progress:

            inputs = (
                batch[
                    "input"
                ]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )

            targets = (
                batch[
                    "target"
                ]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            quality_optimizer.zero_grad(
                set_to_none=True
            )


            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED
            ):

                predictions = (
                    quality_predictor(
                        inputs
                    )
                )

                loss = F.mse_loss(
                    predictions,
                    targets
                )


            quality_scaler.scale(
                loss
            ).backward()


            quality_scaler.unscale_(
                quality_optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                quality_predictor.parameters(),
                max_norm=5.0
            )


            quality_scaler.step(
                quality_optimizer
            )

            quality_scaler.update()


            epoch_losses.append(
                float(
                    loss.item()
                )
            )


            progress.set_postfix({
                "mse":
                    f"{np.mean(epoch_losses):.5f}"
            })


        validation = quality_validation_metrics(
            quality_predictor,
            quality_val_loader
        )


        quality_scheduler.step(
            validation[
                "mae"
            ]
        )


        row = {
            "epoch":
                epoch,

            "train_mse":
                float(
                    np.mean(
                        epoch_losses
                    )
                ),

            "val_mse":
                validation[
                    "loss"
                ],

            "val_mae":
                validation[
                    "mae"
                ],

            "val_rmse":
                validation[
                    "rmse"
                ],

            "val_pearson":
                validation[
                    "pearson"
                ],

            "learning_rate":
                float(
                    quality_optimizer
                    .param_groups[
                        0
                    ][
                        "lr"
                    ]
                ),
        }


        quality_history.append(
            row
        )


        print(
            row
        )


        if (
            validation[
                "mae"
            ]
            < best_quality_mae
        ):

            best_quality_mae = (
                validation[
                    "mae"
                ]
            )

            quality_epochs_without_improvement = 0


            atomic_torch_save(
                {
                    "epoch":
                        epoch,

                    "model_state_dict":
                        quality_predictor
                        .state_dict(),

                    "best_val_mae":
                        best_quality_mae,

                    "config": {
                        "input_channels":
                            2,

                        "lowres_shape":
                            LOWRES_SHAPE,

                        "perturbations":
                            PERTURBATION_KINDS,
                    },
                },
                QUALITY_BEST_CKPT
            )

        else:

            quality_epochs_without_improvement += 1


        atomic_torch_save(
            {
                "epoch":
                    epoch,

                "model_state_dict":
                    quality_predictor
                    .state_dict(),

                "optimizer_state_dict":
                    quality_optimizer
                    .state_dict(),

                "best_val_mae":
                    best_quality_mae,

                "epochs_without_improvement":
                    quality_epochs_without_improvement,

                "history":
                    quality_history,
            },
            QUALITY_LAST_CKPT
        )


        atomic_csv_save(
            pd.DataFrame(
                quality_history
            ),
            DOWNSTREAM_ROOT
            / "quality_predictor_training_history.csv"
        )


        if (
            quality_epochs_without_improvement
            >= QUALITY_EARLY_STOPPING_PATIENCE
        ):

            print(
                "Quality-predictor early stopping triggered."
            )

            break


    atomic_json_dump(
        {
            "completed":
                True,

            "best_validation_mae":
                float(
                    best_quality_mae
                ),

            "best_checkpoint":
                str(
                    QUALITY_BEST_CKPT
                ),

            "last_epoch":
                int(
                    quality_history[
                        -1
                    ][
                        "epoch"
                    ]
                ),
        },
        QUALITY_DONE_JSON
    )

else:

    print(
        "Quality-predictor training already completed -> skipped"
    )


if not QUALITY_BEST_CKPT.exists():

    raise FileNotFoundError(
        "Best quality-predictor checkpoint was not created."
    )


best_quality_checkpoint = torch.load(
    QUALITY_BEST_CKPT,
    map_location=DEVICE
)


quality_predictor.load_state_dict(
    best_quality_checkpoint[
        "model_state_dict"
    ]
)


quality_predictor.eval()


print(
    "Loaded best quality-predictor validation MAE:",
    best_quality_checkpoint[
        "best_val_mae"
    ]
)


Quality epoch 1/40:   0%|          | 0/520 [00:00<?, ?it/s]

{'epoch': 1, 'train_mse': 0.022945349464650742, 'val_mse': 0.037328438544108604, 'val_mae': 0.14635703705537778, 'val_rmse': 0.19320569090130382, 'val_pearson': 0.4138722241560384, 'learning_rate': 0.0001}


Quality epoch 2/40:   0%|          | 0/520 [00:00<?, ?it/s]

{'epoch': 2, 'train_mse': 0.020516514072033504, 'val_mse': 0.032859471521155335, 'val_mae': 0.13613355760104381, 'val_rmse': 0.18127181736816392, 'val_pearson': 0.5802496181399857, 'learning_rate': 0.0001}


Quality epoch 3/40:   0%|          | 0/520 [00:00<?, ?it/s]

...

{'epoch': 38, 'train_mse': 0.0016724300314808333, 'val_mse': 0.009724082555168514, 'val_mae': 0.06622712937398599, 'val_rmse': 0.0986107627046529, 'val_pearson': 0.8685012584207186, 'learning_rate': 6.25e-06}


Quality epoch 39/40:   0%|          | 0/520 [00:00<?, ?it/s]

{'epoch': 39, 'train_mse': 0.0017205104244108218, 'val_mse': 0.010088046570504397, 'val_mae': 0.06659782808274031, 'val_rmse': 0.10043926783018768, 'val_pearson': 0.8702519685292059, 'learning_rate': 6.25e-06}


Quality epoch 40/40:   0%|          | 0/520 [00:00<?, ?it/s]

{'epoch': 40, 'train_mse': 0.0016716132221114813, 'val_mse': 0.009562059741699613, 'val_mae': 0.06533530175399321, 'val_rmse': 0.09778578470164946, 'val_pearson': 0.8728487245136303, 'learning_rate': 3.125e-06}
Loaded best quality-predictor validation MAE: 0.0648025820318323


In [17]:
quality_validation = quality_validation_metrics(
    quality_predictor,
    quality_val_loader
)


quality_validation_rows = []


row_index = 0


for batch in quality_val_loader:

    batch_size = (
        batch[
            "target"
        ].shape[
            0
        ]
    )


    for local_index in range(
        batch_size
    ):

        quality_validation_rows.append({
            "subject":
                batch[
                    "subject"
                ][
                    local_index
                ],

            "perturbation":
                batch[
                    "perturbation"
                ][
                    local_index
                ],

            "true_dice":
                float(
                    quality_validation[
                        "targets"
                    ][
                        row_index
                    ]
                ),

            "predicted_dice":
                float(
                    quality_validation[
                        "predictions"
                    ][
                        row_index
                    ]
                ),

            "absolute_error":
                float(
                    abs(
                        quality_validation[
                            "predictions"
                        ][
                            row_index
                        ]
                        - quality_validation[
                            "targets"
                        ][
                            row_index
                        ]
                    )
                ),
        })


        row_index += 1


quality_validation_df = pd.DataFrame(
    quality_validation_rows
)


atomic_csv_save(
    quality_validation_df,
    DOWNSTREAM_ROOT
    / "quality_predictor_internal_validation.csv"
)


print({
    "validation_mae":
        quality_validation[
            "mae"
        ],

    "validation_rmse":
        quality_validation[
            "rmse"
        ],

    "validation_pearson":
        quality_validation[
            "pearson"
        ],
})


display(
    quality_validation_df.head()
)


{'validation_mae': 0.0648025820318323, 'validation_rmse': 0.09746271302723727, 'validation_pearson': 0.8743910151928528}


,subject,perturbation,true_dice,predicted_dice,absolute_error
0,BraTS-GLI-00751-000,baseline,0.913531,0.906738,0.006792
1,BraTS-GLI-00751-000,erode_1,0.830259,0.837891,0.007631
2,BraTS-GLI-00751-000,erode_2,0.697309,0.703125,0.005816
3,BraTS-GLI-00751-000,erode_3,0.563155,0.564453,0.001298
4,BraTS-GLI-00751-000,dilate_1,0.848633,0.869141,0.020508


In [18]:
segmentor = (
    WholeTumourUNet3D(
        base_channels=16
    )
    .to(
        DEVICE
    )
)


segmentor.load_state_dict(
    torch.load(
        SEGMENTOR_BEST_CKPT,
        map_location=DEVICE
    )[
        "model_state_dict"
    ]
)


segmentor.eval()

quality_predictor.to(
    DEVICE
)

quality_predictor.eval()


heldout_quality_rows = []


for subject in tqdm(
    evaluation_subjects,
    desc="Held-out real EvanySeg validation"
):

    with np.load(
        REAL_LOWRES_DIR
        / f"{subject}.npz"
    ) as cached:

        image = cached[
            "image"
        ].astype(
            np.float32
        )

        ground_truth = cached[
            "mask"
        ].astype(
            np.uint8
        )


    _, prediction = (
        predict_lowres_whole_tumour(
            segmentor,
            image
        )
    )


    quality_input = (
        torch.from_numpy(
            np.stack(
                [
                    image,
                    prediction.astype(
                        np.float32
                    )
                ],
                axis=0
            )
        )
        .unsqueeze(0)
        .to(
            DEVICE
        )
    )


    with torch.inference_mode():

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=AMP_ENABLED
        ):

            predicted_dice = float(
                quality_predictor(
                    quality_input
                )[
                    0
                ]
                .float()
                .cpu()
                .item()
            )


    actual_dice = dice_np(
        prediction,
        ground_truth
    )


    heldout_quality_rows.append({
        "subject":
            subject,

        "actual_dice":
            actual_dice,

        "predicted_dice":
            predicted_dice,

        "absolute_error":
            abs(
                predicted_dice
                - actual_dice
            ),
    })


heldout_quality_df = pd.DataFrame(
    heldout_quality_rows
)


atomic_csv_save(
    heldout_quality_df,
    DOWNSTREAM_ROOT
    / "quality_predictor_external_heldout_200.csv"
)


heldout_mae = float(
    heldout_quality_df[
        "absolute_error"
    ].mean()
)


heldout_rmse = float(
    np.sqrt(
        np.mean(
            (
                heldout_quality_df[
                    "predicted_dice"
                ]
                - heldout_quality_df[
                    "actual_dice"
                ]
            ) ** 2
        )
    )
)


if (
    heldout_quality_df[
        "predicted_dice"
    ].std() > 0
    and heldout_quality_df[
        "actual_dice"
    ].std() > 0
):

    heldout_pearson = float(
        pearsonr(
            heldout_quality_df[
                "predicted_dice"
            ],
            heldout_quality_df[
                "actual_dice"
            ]
        ).statistic
    )

else:

    heldout_pearson = np.nan


print({
    "heldout_200_mae":
        heldout_mae,

    "heldout_200_rmse":
        heldout_rmse,

    "heldout_200_pearson":
        heldout_pearson,
})


if heldout_mae > 0.15:

    warnings.warn(
        "The external quality-predictor MAE exceeds 0.15. "
        "Interpret synthetic EvanySeg-style scores cautiously."
    )


segmentor.to(
    "cpu"
)

quality_predictor.to(
    "cpu"
)

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


Held-out real EvanySeg validation:   0%|          | 0/200 [00:00<?, ?it/s]

{'heldout_200_mae': 0.043543310816334724, 'heldout_200_rmse': 0.06291101660294739, 'heldout_200_pearson': 0.8273615222990427}


In [19]:
def extract_sample_id(
    filename: str
) -> str:

    match = re.search(
        r"(\d{4})(?=\.nii\.gz$)",
        filename
    )


    if match is None:

        raise ValueError(
            f"Could not extract sample ID from {filename}"
        )


    return match.group(
        1
    )


synthetic_manifest_rows = []


for model_name, directory in (
    MODEL_DIRS.items()
):

    prefix = MODEL_PREFIXES[
        model_name
    ]


    files = sorted(
        directory.glob(
            f"{prefix}_*.nii.gz"
        )
    )


    if len(files) != 200:

        raise RuntimeError(
            f"{model_name} must contain 200 source volumes; "
            f"found {len(files)}."
        )


    cache_directory = (
        SYNTHETIC_LOWRES_ROOT
        / model_name
    )


    for path in tqdm(
        files,
        desc=f"Verify synthetic cache: {model_name}"
    ):

        sample_id = extract_sample_id(
            path.name
        )


        cache_path = (
            cache_directory
            / f"{sample_id}.npz"
        )


        if not cache_path.exists():

            raise FileNotFoundError(
                "The shared synthetic low-resolution cache "
                f"is missing: {cache_path}"
            )


        synthetic_manifest_rows.append({
            "model":
                model_name,

            "sample_id":
                sample_id,

            "source_filename":
                path.name,

            "source_path":
                str(
                    path
                ),

            "lowres_cache":
                str(
                    cache_path
                ),

            "run_label":
                RUN_LABEL,

            "training_seed":
                TRAINING_SEED,
        })


synthetic_manifest_df = pd.DataFrame(
    synthetic_manifest_rows
)


if len(
    synthetic_manifest_df
) != 600:

    raise RuntimeError(
        "Expected 600 synthetic records."
    )


atomic_csv_save(
    synthetic_manifest_df,
    DOWNSTREAM_ROOT
    / "synthetic_downstream_manifest.csv"
)


display(
    synthetic_manifest_df
    .groupby(
        "model"
    )
    .size()
    .rename(
        "volumes"
    )
    .reset_index()
)


print(
    "Shared synthetic cache verified."
)

print(
    "No shared cache files will be modified by this run."
)


Verify synthetic cache: ddpm_v5:   0%|          | 0/200 [00:00<?, ?it/s]

Verify synthetic cache: conditional_ddpm_v3:   0%|          | 0/200 [00:00<?, ?it/s]

Verify synthetic cache: conditional_ldm_v4:   0%|          | 0/200 [00:00<?, ?it/s]

,model,volumes
0,conditional_ddpm_v3,200
1,conditional_ldm_v4,200
2,ddpm_v5,200


Shared synthetic cache verified.
No shared cache files will be modified by this run.


In [20]:
segmentor = (
    WholeTumourUNet3D(
        base_channels=16
    )
    .to(
        DEVICE
    )
)


segmentor.load_state_dict(
    torch.load(
        SEGMENTOR_BEST_CKPT,
        map_location=DEVICE
    )[
        "model_state_dict"
    ]
)


segmentor.eval()


for model_name in MODEL_DIRS:

    model_manifest = (
        synthetic_manifest_df[
            synthetic_manifest_df[
                "model"
            ]
            == model_name
        ]
        .sort_values(
            "sample_id"
        )
    )


    lowres_prediction_dir = (
        SYNTHETIC_PRED_LOWRES_ROOT
        / model_name
    )


    full_prediction_dir = (
        PREDICTED_MASK_ROOT
        / model_name
    )


    lowres_prediction_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    full_prediction_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    for row in tqdm(
        model_manifest.itertuples(
            index=False
        ),
        total=len(
            model_manifest
        ),
        desc=f"WT inference: {model_name}"
    ):

        lowres_output_path = (
            lowres_prediction_dir
            / f"{row.sample_id}.npz"
        )


        full_output_path = (
            full_prediction_dir
            / (
                f"{model_name}_wt_pred_"
                f"{row.sample_id}.nii.gz"
            )
        )


        if (
            lowres_output_path.exists()
            and full_output_path.exists()
            and not FORCE_SYNTHETIC_INFERENCE
        ):

            continue


        with np.load(
            row.lowres_cache
        ) as cached:

            image = cached[
                "image"
            ].astype(
                np.float32
            )


        probability, prediction = (
            predict_lowres_whole_tumour(
                segmentor,
                image
            )
        )


        np.savez(
            lowres_output_path,
            probability=probability.astype(
                np.float16
            ),
            prediction=prediction.astype(
                np.uint8
            ),
        )


        full_prediction = resize_mask_np(
            prediction,
            output_shape=FULL_SHAPE
        )


        nifti = nib.Nifti1Image(
            full_prediction.astype(
                np.uint8
            ),
            np.eye(
                4,
                dtype=np.float32
            )
        )


        nifti.set_data_dtype(
            np.uint8
        )


        nib.save(
            nifti,
            full_output_path
        )


segmentor.to(
    "cpu"
)

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


print(
    "Synthetic WT segmentation inference complete."
)


WT inference: ddpm_v5:   0%|          | 0/200 [00:00<?, ?it/s]

WT inference: conditional_ddpm_v3:   0%|          | 0/200 [00:00<?, ?it/s]

WT inference: conditional_ldm_v4:   0%|          | 0/200 [00:00<?, ?it/s]

Synthetic WT segmentation inference complete.


In [21]:
quality_predictor = (
    WholeTumourQualityRegressor3D()
    .to(
        DEVICE
    )
)


quality_predictor.load_state_dict(
    torch.load(
        QUALITY_BEST_CKPT,
        map_location=DEVICE
    )[
        "model_state_dict"
    ]
)


quality_predictor.eval()


all_downstream_rows = []


for model_name in MODEL_DIRS:

    model_output_csv = (
        DOWNSTREAM_ROOT
        / f"{model_name}_downstream_per_sample.csv"
    )


    existing_rows = {}


    if (
        model_output_csv.exists()
        and not FORCE_SYNTHETIC_INFERENCE
    ):

        existing_df = pd.read_csv(
            model_output_csv,
            dtype={
                "sample_id":
                    str
            }
        )


        for row in existing_df.to_dict(
            orient="records"
        ):

            existing_rows[
                str(
                    row[
                        "sample_id"
                    ]
                ).zfill(
                    4
                )
            ] = row


    model_manifest = (
        synthetic_manifest_df[
            synthetic_manifest_df[
                "model"
            ]
            == model_name
        ]
        .sort_values(
            "sample_id"
        )
    )


    model_rows = []


    for row in tqdm(
        model_manifest.itertuples(
            index=False
        ),
        total=len(
            model_manifest
        ),
        desc=f"WT quality scoring: {model_name}"
    ):

        sample_id = str(
            row.sample_id
        ).zfill(
            4
        )


        if (
            sample_id
            in existing_rows
            and not FORCE_SYNTHETIC_INFERENCE
        ):

            model_rows.append(
                existing_rows[
                    sample_id
                ]
            )

            continue


        with np.load(
            row.lowres_cache
        ) as cached:

            image = cached[
                "image"
            ].astype(
                np.float32
            )


        prediction_path = (
            SYNTHETIC_PRED_LOWRES_ROOT
            / model_name
            / f"{sample_id}.npz"
        )


        with np.load(
            prediction_path
        ) as cached_prediction:

            prediction = (
                cached_prediction[
                    "prediction"
                ]
                .astype(
                    np.uint8
                )
            )

            probability = (
                cached_prediction[
                    "probability"
                ]
                .astype(
                    np.float32
                )
            )


        paired_input = (
            torch.from_numpy(
                np.stack(
                    [
                        image,
                        prediction.astype(
                            np.float32
                        )
                    ],
                    axis=0
                )
            )
            .unsqueeze(0)
            .to(
                DEVICE
            )
        )


        with torch.inference_mode():

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED
            ):

                predicted_dice = float(
                    quality_predictor(
                        paired_input
                    )[
                        0
                    ]
                    .float()
                    .cpu()
                    .item()
                )


        condition_dice = np.nan


        if model_name in {
            "conditional_ddpm_v3",
            "conditional_ldm_v4"
        }:

            condition_path = (
                CONDITION_MASK_DIR
                / (
                    f"condition_mask_"
                    f"{sample_id}.nii.gz"
                )
            )


            if not condition_path.exists():

                raise FileNotFoundError(
                    f"Missing condition mask: {condition_path}"
                )


            condition_full = (
                np.asarray(
                    nib.load(
                        condition_path
                    ).dataobj
                )
                > 0
            ).astype(
                np.uint8
            )


            condition_low = resize_mask_np(
                condition_full
            )


            condition_dice = dice_np(
                prediction,
                condition_low
            )


        tumour_probabilities = probability[
            prediction > 0
        ]


        model_rows.append({
            "model":
                model_name,

            "run_label":
                RUN_LABEL,

            "training_seed":
                TRAINING_SEED,

            "sample_id":
                sample_id,

            "source_filename":
                row.source_filename,

            "wt_evanyseg_predicted_dice":
                predicted_dice,

            "condition_adherence_dice":
                condition_dice,

            "predicted_tumour_voxels_lowres":
                int(
                    prediction.sum()
                ),

            "predicted_tumour_fraction_lowres":
                float(
                    prediction.mean()
                ),

            "mean_probability_inside_prediction":
                float(
                    tumour_probabilities.mean()
                )
                if len(
                    tumour_probabilities
                ) > 0
                else 0.0,

            "empty_prediction":
                bool(
                    prediction.sum()
                    == 0
                ),
        })


        atomic_csv_save(
            pd.DataFrame(
                model_rows
            ),
            model_output_csv
        )


    model_df = pd.DataFrame(
        model_rows
    ).sort_values(
        "sample_id"
    )


    atomic_csv_save(
        model_df,
        model_output_csv
    )


    all_downstream_rows.extend(
        model_df.to_dict(
            orient="records"
        )
    )


downstream_per_sample_df = pd.DataFrame(
    all_downstream_rows
)


atomic_csv_save(
    downstream_per_sample_df,
    DOWNSTREAM_ROOT
    / "downstream_per_sample_all_models.csv"
)


quality_predictor.to(
    "cpu"
)

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


print(
    "Synthetic WT downstream scoring complete."
)


WT quality scoring: ddpm_v5:   0%|          | 0/200 [00:00<?, ?it/s]

WT quality scoring: conditional_ddpm_v3:   0%|          | 0/200 [00:00<?, ?it/s]

WT quality scoring: conditional_ldm_v4:   0%|          | 0/200 [00:00<?, ?it/s]

Synthetic WT downstream scoring complete.


In [22]:
def bootstrap_mean_ci(
    values,
    repeats=10000,
    confidence=0.95,
    seed=BOOTSTRAP_SEED
):

    values = np.asarray(
        values,
        dtype=np.float64
    )


    rng = np.random.default_rng(
        seed
    )


    bootstrap_means = np.empty(
        repeats,
        dtype=np.float64
    )


    for index in range(
        repeats
    ):

        bootstrap_sample = rng.choice(
            values,
            size=len(
                values
            ),
            replace=True
        )


        bootstrap_means[
            index
        ] = np.mean(
            bootstrap_sample
        )


    alpha = (
        1.0
        - confidence
    )


    lower = float(
        np.quantile(
            bootstrap_means,
            alpha
            / 2.0
        )
    )


    upper = float(
        np.quantile(
            bootstrap_means,
            1.0
            - alpha
            / 2.0
        )
    )


    return (
        lower,
        upper
    )


summary_rows = []


for model_name in MODEL_DIRS:

    model_df = downstream_per_sample_df[
        downstream_per_sample_df[
            "model"
        ]
        == model_name
    ].copy()


    evanyseg_values = (
        model_df[
            "wt_evanyseg_predicted_dice"
        ]
        .astype(
            float
        )
        .to_numpy()
    )


    ci_lower, ci_upper = (
        bootstrap_mean_ci(
            evanyseg_values
        )
    )


    condition_values = (
        model_df[
            "condition_adherence_dice"
        ]
        .dropna()
        .astype(
            float
        )
        .to_numpy()
    )


    summary_rows.append({
        "model":
            model_name,

        "run_label":
            RUN_LABEL,

        "training_seed":
            TRAINING_SEED,

        "model_display_name":
            MODEL_DISPLAY_NAMES[
                model_name
            ],

        "n_volumes":
            len(
                model_df
            ),

        "wt_evanyseg_mean":
            float(
                np.mean(
                    evanyseg_values
                )
            ),

        "wt_evanyseg_std":
            float(
                np.std(
                    evanyseg_values,
                    ddof=1
                )
            ),

        "wt_evanyseg_median":
            float(
                np.median(
                    evanyseg_values
                )
            ),

        "wt_evanyseg_ci95_lower":
            ci_lower,

        "wt_evanyseg_ci95_upper":
            ci_upper,

        "condition_adherence_dice_mean":
            float(
                np.mean(
                    condition_values
                )
            )
            if len(
                condition_values
            ) > 0
            else np.nan,

        "condition_adherence_dice_std":
            float(
                np.std(
                    condition_values,
                    ddof=1
                )
            )
            if len(
                condition_values
            ) > 1
            else np.nan,

        "empty_prediction_percentage":
            float(
                model_df[
                    "empty_prediction"
                ]
                .astype(
                    bool
                )
                .mean()
                * 100.0
            ),
    })


downstream_summary_df = pd.DataFrame(
    summary_rows
).sort_values(
    "wt_evanyseg_mean",
    ascending=False
)


atomic_csv_save(
    downstream_summary_df,
    DOWNSTREAM_ROOT
    / "downstream_summary.csv"
)


downstream_results_payload = {
    row[
        "model"
    ]: {
        key:
            (
                None
                if pd.isna(
                    value
                )
                else (
                    value.item()
                    if isinstance(
                        value,
                        np.generic
                    )
                    else value
                )
            )
        for key, value in row.items()
        if key not in {
            "model"
        }
    }
    for row in summary_rows
}


atomic_json_dump(
    downstream_results_payload,
    DOWNSTREAM_ROOT
    / "downstream_results.json"
)


display(
    downstream_summary_df
)


,model,run_label,training_seed,model_display_name,n_volumes,wt_evanyseg_mean,wt_evanyseg_std,wt_evanyseg_median,wt_evanyseg_ci95_lower,wt_evanyseg_ci95_upper,condition_adherence_dice_mean,condition_adherence_dice_std,empty_prediction_percentage
1,conditional_ddpm_v3,v3,2028,Conditional DDPM V3,200,0.871768,0.044852,0.880127,0.865559,0.877712,0.686752,0.254199,0.0
2,conditional_ldm_v4,v3,2028,Conditional LDM V4,200,0.863721,0.060926,0.879883,0.855007,0.871714,0.665182,0.242862,0.0
0,ddpm_v5,v3,2028,DDPM V5,200,0.788079,0.070283,0.796143,0.778286,0.797722,NaN,NaN,0.0


In [23]:
UPSTREAM_RESULTS_JSON = (
    PROJECT_ROOT
    / "upstream_results"
    / "upstream_results.json"
)


if not UPSTREAM_RESULTS_JSON.exists():

    raise FileNotFoundError(
        f"Upstream result JSON not found: "
        f"{UPSTREAM_RESULTS_JSON}"
    )


with open(
    UPSTREAM_RESULTS_JSON,
    "r"
) as f:

    upstream_results = json.load(f)


UPSTREAM_METRIC_DIRECTIONS = {
    "FID":
        "lower",

    "KID":
        "lower",

    "sFID":
        "lower",

    "Precision":
        "higher",

    "Recall":
        "higher",

    "Density":
        "higher",

    "Coverage":
        "higher",

    "FRD":
        "lower",

    "RadFID":
        "lower",

    "MedFID":
        "lower",

    "Vendi":
        "higher",

    "AuthPct":
        "higher",

    "ASW":
        "lower",
}


comparison_rows = []


summary_indexed = (
    downstream_summary_df
    .set_index(
        "model"
    )
)


for model_name in MODEL_DIRS:

    row = {
        "model":
            model_name,

        "run_label":
            RUN_LABEL,

        "training_seed":
            TRAINING_SEED,

        "model_display_name":
            MODEL_DISPLAY_NAMES[
                model_name
            ],

        "WT_EvanySeg":
            float(
                summary_indexed.loc[
                    model_name,
                    "wt_evanyseg_mean"
                ]
            ),

        "WT_EvanySeg_CI95_lower":
            float(
                summary_indexed.loc[
                    model_name,
                    "wt_evanyseg_ci95_lower"
                ]
            ),

        "WT_EvanySeg_CI95_upper":
            float(
                summary_indexed.loc[
                    model_name,
                    "wt_evanyseg_ci95_upper"
                ]
            ),

        "WT_condition_adherence_Dice":
            summary_indexed.loc[
                model_name,
                "condition_adherence_dice_mean"
            ],
    }


    model_upstream = (
        upstream_results.get(
            model_name,
            {}
        )
    )


    for metric in (
        UPSTREAM_METRIC_DIRECTIONS
    ):

        row[
            metric
        ] = model_upstream.get(
            metric,
            np.nan
        )


    comparison_rows.append(
        row
    )


upstream_downstream_df = pd.DataFrame(
    comparison_rows
)


atomic_csv_save(
    upstream_downstream_df,
    DOWNSTREAM_ROOT
    / "upstream_downstream_comparison.csv"
)


alignment_rows = []


downstream_scores = (
    upstream_downstream_df[
        "WT_EvanySeg"
    ]
    .astype(
        float
    )
    .to_numpy()
)


downstream_order = list(
    upstream_downstream_df.iloc[
        np.argsort(
            -downstream_scores
        )
    ][
        "model"
    ]
)


for metric, direction in (
    UPSTREAM_METRIC_DIRECTIONS.items()
):

    metric_values = pd.to_numeric(
        upstream_downstream_df[
            metric
        ],
        errors="coerce"
    ).to_numpy(
        dtype=float
    )


    valid = np.isfinite(
        metric_values
    )


    if valid.sum() != 3:

        alignment_rows.append({
            "metric":
                metric,

            "direction":
                direction,

            "spearman_rho":
                np.nan,

            "ranking_match":
                False,

            "metric_ranking":
                "incomplete",

            "downstream_ranking":
                " > ".join(
                    downstream_order
                ),

            "note":
                "Metric missing for at least one model.",
        })

        continue


    quality_oriented = (
        metric_values
        if direction
        == "higher"
        else -metric_values
    )


    rho = float(
        spearmanr(
            quality_oriented,
            downstream_scores
        ).statistic
    )


    metric_order = list(
        upstream_downstream_df.iloc[
            np.argsort(
                -quality_oriented
            )
        ][
            "model"
        ]
    )


    alignment_rows.append({
        "metric":
            metric,

        "direction":
            direction,

        "spearman_rho":
            rho,

        "ranking_match":
            metric_order
            == downstream_order,

        "metric_ranking":
            " > ".join(
                metric_order
            ),

        "downstream_ranking":
            " > ".join(
                downstream_order
            ),

        "note":
            (
                "Descriptive only: the correlation is based "
                "on three generative models."
            ),
    })


alignment_df = pd.DataFrame(
    alignment_rows
)


atomic_csv_save(
    alignment_df,
    DOWNSTREAM_ROOT
    / "upstream_downstream_rank_alignment.csv"
)


display(
    upstream_downstream_df
)


display(
    alignment_df
)


/tmp/ipykernel_2010404/583431284.py:246: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearmanr(


,model,run_label,training_seed,model_display_name,WT_EvanySeg,WT_EvanySeg_CI95_lower,WT_EvanySeg_CI95_upper,WT_condition_adherence_Dice,FID,KID,...,Precision,Recall,Density,Coverage,FRD,RadFID,MedFID,Vendi,AuthPct,ASW
0,ddpm_v5,v3,2028,DDPM V5,0.788079,0.778286,0.797722,NaN,118.095077,0.115825,...,0.750,0.24,0.207,0.105,58.144048,0.0,0.686781,2.229131,92.0,0.487527
1,conditional_ddpm_v3,v3,2028,Conditional DDPM V3,0.871768,0.865559,0.877712,0.686752,146.471241,0.149752,...,0.885,0.38,0.292,0.160,58.112696,0.0,0.446472,2.868546,87.0,0.398013
2,conditional_ldm_v4,v3,2028,Conditional LDM V4,0.863721,0.855007,0.871714,0.665182,72.989703,0.052913,...,1.000,0.79,1.224,0.865,51.152709,0.0,0.128688,3.427445,69.5,0.169103


,metric,direction,spearman_rho,ranking_match,metric_ranking,downstream_ranking,note
0,FID,lower,-0.5,False,conditional_ldm_v4 > ddpm_v5 > conditional_ddp...,conditional_ddpm_v3 > conditional_ldm_v4 > ddp...,Descriptive only: the correlation is based on ...
1,KID,lower,-0.5,False,conditional_ldm_v4 > ddpm_v5 > conditional_ddp...,conditional_ddpm_v3 > conditional_ldm_v4 > ddp...,Descriptive only: the correlation is based on ...
2,sFID,lower,-0.5,False,conditional_ldm_v4 > ddpm_v5 > conditional_ddp...,conditional_ddpm_v3 > conditional_ldm_v4 > ddp...,Descriptive only: the correlation is based on ...
3,Precision,higher,0.5,False,conditional_ldm_v4 > conditional_ddpm_v3 > ddp...,conditional_ddpm_v3 > conditional_ldm_v4 > ddp...,Descriptive only: the correlation is based on ...
4,Recall,higher,0.5,False,conditional_ldm_v4 > conditional_ddpm_v3 > ddp...,conditional_ddpm_v3 > conditional_ldm_v4 > ddp...,Descriptive only: the correlation is based on ...
5,Density,higher,0.5,False,conditional_ldm_v4 > conditional_ddpm_v3 > ddp...,conditional_ddpm_v3 > conditional_ldm_v4 > ddp...,Descriptive only: the correlation is based on ...
6,Coverage,higher,0.5,False,conditional_ldm_v4 > conditional_ddpm_v3 > ddp...,conditional_ddpm_v3 > conditional_ldm_v4 > ddp...,Descriptive only: the correlation is based on ...
7,FRD,lower,0.5,False,conditional_ldm_v4 > conditional_ddpm_v3 > ddp...,conditional_ddpm_v3 > conditional_ldm_v4 > ddp...,Descriptive only: the correlation is based on ...
8,RadFID,lower,NaN,False,ddpm_v5 > conditional_ddpm_v3 > conditional_ld...,conditional_ddpm_v3 > conditional_ldm_v4 > ddp...,Descriptive only: the correlation is based on ...
9,MedFID,lower,0.5,False,conditional_ldm_v4 > conditional_ddpm_v3 > ddp...,conditional_ddpm_v3 > conditional_ldm_v4 > ddp...,Descriptive only: the correlation is based on ...


In [24]:
downstream_protocol = {
    "task":
        "Binary Whole-Tumour Segmentation",

    "modality":
        "BraTS 2023 T2-FLAIR",

    "whole_tumour_definition":
        "All non-background BraTS segmentation labels (seg > 0)",

    "full_volume_shape":
        list(
            FULL_SHAPE
        ),

    "downstream_training_shape":
        list(
            LOWRES_SHAPE
        ),

    "segmentor": {
        "architecture":
            "Custom 3D U-Net",

        "training_subjects":
            len(
                seg_train_subjects
            ),

        "validation_subjects":
            len(
                seg_val_subjects
            ),

        "threshold":
            SEGMENTATION_THRESHOLD,

        "best_checkpoint":
            str(
                SEGMENTOR_BEST_CKPT
            ),
    },

    "quality_predictor": {
        "architecture":
            "Compact 3D ResNet Dice regressor",

        "input":
            "Two channels: T2-FLAIR image and predicted/perturbed WT mask",

        "training_subjects":
            len(
                quality_train_subjects
            ),

        "validation_subjects":
            len(
                quality_val_subjects
            ),

        "perturbations":
            PERTURBATION_KINDS,

        "target":
            "True Dice against the real whole-tumour mask",

        "best_checkpoint":
            str(
                QUALITY_BEST_CKPT
            ),
    },

    "main_downstream_metric":
        "Mean WT EvanySeg-style predicted Dice across 200 synthetic volumes",

    "secondary_conditional_metric":
        (
            "Dice between the fixed segmentor prediction "
            "and the generation condition mask; reported only "
            "for the two conditional generators."
        ),

    "evaluation_models":
        list(
            MODEL_DIRS.keys()
        ),

    "synthetic_volumes_per_model":
        200,

    "training_evaluation_leakage":
        "None: all downstream training subsets come from data_split['train']; the fixed 200 synthetic evaluation conditions are excluded.",

    "run_label":
        RUN_LABEL,

    "split_seed":
        SPLIT_SEED,

    "training_seed":
        TRAINING_SEED,

    "perturbation_seed":
        PERTURBATION_SEED,

    "bootstrap_seed":
        BOOTSTRAP_SEED,

    "run_root":
        str(
            RUN_ROOT
        ),

    "rank_alignment_note":
        "Upstream/downstream Spearman correlations are descriptive because only three generative models are compared.",
}


atomic_json_dump(
    downstream_protocol,
    DOWNSTREAM_ROOT
    / "downstream_evaluation_protocol.json"
)


print(
    "Downstream protocol saved."
)


Downstream protocol saved.


In [25]:
required_files = {
    "segmentor checkpoint":
        SEGMENTOR_BEST_CKPT,

    "quality-predictor checkpoint":
        QUALITY_BEST_CKPT,

    "segmentor real performance":
        DOWNSTREAM_ROOT
        / "real_whole_tumour_segmentor_performance.csv",

    "quality internal validation":
        DOWNSTREAM_ROOT
        / "quality_predictor_internal_validation.csv",

    "quality external validation":
        DOWNSTREAM_ROOT
        / "quality_predictor_external_heldout_200.csv",

    "per-sample downstream results":
        DOWNSTREAM_ROOT
        / "downstream_per_sample_all_models.csv",

    "downstream summary":
        DOWNSTREAM_ROOT
        / "downstream_summary.csv",

    "downstream JSON":
        DOWNSTREAM_ROOT
        / "downstream_results.json",

    "upstream/downstream comparison":
        DOWNSTREAM_ROOT
        / "upstream_downstream_comparison.csv",

    "rank alignment":
        DOWNSTREAM_ROOT
        / "upstream_downstream_rank_alignment.csv",

    "protocol":
        DOWNSTREAM_ROOT
        / "downstream_evaluation_protocol.json",
}


audit_rows = [
    {
        "artifact":
            name,

        "path":
            str(
                path
            ),

        "exists":
            path.exists(),
    }
    for name, path in (
        required_files.items()
    )
]


audit_df = pd.DataFrame(
    audit_rows
)


display(
    audit_df
)


if not audit_df[
    "exists"
].all():

    missing = audit_df[
        ~audit_df[
            "exists"
        ]
    ][
        "artifact"
    ].tolist()


    raise RuntimeError(
        f"Downstream evaluation is incomplete: {missing}"
    )


if len(
    downstream_per_sample_df
) != 600:

    raise RuntimeError(
        "Per-sample downstream results must contain 600 rows."
    )


if not (
    downstream_per_sample_df
    .groupby(
        "model"
    )
    .size()
    == 200
).all():

    raise RuntimeError(
        "Each model must have exactly 200 downstream scores."
    )


print()
print(
    "========================================"
)

print(
    "Whole-Tumour downstream evaluation complete"
)

print(
    "========================================"
)

print(
    "Run label:",
    RUN_LABEL
)

print(
    "Training seed:",
    TRAINING_SEED
)

print(
    "Results directory:",
    DOWNSTREAM_ROOT
)


,artifact,path,exists
0,segmentor checkpoint,/users/tdd540/Capstone_Project/downstream_v3/c...,True
1,quality-predictor checkpoint,/users/tdd540/Capstone_Project/downstream_v3/c...,True
2,segmentor real performance,/users/tdd540/Capstone_Project/downstream_v3/r...,True
3,quality internal validation,/users/tdd540/Capstone_Project/downstream_v3/r...,True
4,quality external validation,/users/tdd540/Capstone_Project/downstream_v3/r...,True
5,per-sample downstream results,/users/tdd540/Capstone_Project/downstream_v3/r...,True
6,downstream summary,/users/tdd540/Capstone_Project/downstream_v3/r...,True
7,downstream JSON,/users/tdd540/Capstone_Project/downstream_v3/r...,True
8,upstream/downstream comparison,/users/tdd540/Capstone_Project/downstream_v3/r...,True
9,rank alignment,/users/tdd540/Capstone_Project/downstream_v3/r...,True



Whole-Tumour downstream evaluation complete
Run label: v3
Training seed: 2028
Results directory: /users/tdd540/Capstone_Project/downstream_v3/results
